# ThyroidGraphNet - HQ profile on Colab
Set **Runtime > Change runtime type > GPU** before running.
Upload the TN5000 zip to Google Drive and set `DATA_ZIP` below.

In [ ]:
REPO = "https://github.com/Ayushii-Kumari/ThyroidGraphNet.git"
DATA_ZIP = "/content/drive/MyDrive/TN5000.zip"
BACKUP = "/content/drive/MyDrive/thyroid_hq"
SEEDS = [42, 43, 44]

import torch
assert torch.cuda.is_available(), "No GPU: change the runtime type to GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q $REPO /content/ThyroidGraphNet
%cd /content/ThyroidGraphNet
# keep Colab's CUDA torch; requirements.txt pins a CPU-only build
!pip install -q timm==1.0.15 scikit-learn pandas matplotlib tqdm pillow

In [ ]:
!mkdir -p data && unzip -q -o "$DATA_ZIP" -d data/
!find data -maxdepth 3 -type d | head -20
import os
assert os.path.isdir("data/TN5000/JPEGImages"), "Move the extracted folder so that data/TN5000/JPEGImages exists"

## Train (one seed per run; checkpoints are backed up to Drive after each)
Already-trained seeds are skipped, so you can re-run this cell after a disconnect.
If you hit CUDA out-of-memory, lower `BATCH_SIZE` in `ConfigHQ` (config.py).

In [ ]:
import os
os.makedirs(BACKUP, exist_ok=True)
!cp -rn $BACKUP/models . 2>/dev/null; true

for seed in SEEDS:
    if os.path.exists(f"models/hq/seed{seed}/thyroidnet_best.pth"):
        print(f"seed {seed}: already trained, skipping")
        continue
    !python train.py --profile hq --seed $seed
    !mkdir -p $BACKUP && cp -r models $BACKUP/

## Evaluate the ensemble (threshold chosen on validation only)

In [ ]:
!python evaluate.py --profile hq
!cp -r results $BACKUP/
!cat results/hq/metrics.json

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix", "roc_curve", "calibration"]:
    display(Image(f"results/hq/{f}.png"))